In [1]:
from sedona.spark import *
from pyspark.sql.functions import *
from pyspark.sql.utils import AnalysisException

config = SedonaContext.builder() \
    .getOrCreate()

sedona = SedonaContext.create(config)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
Setting Spark log level to "WARN".
26/03/02 18:52:04 INFO core/src/lib.rs: Sedona native acceleration engine v0.12.0 ready
                                                                                

# Recommended runtime

## Tiny

# Cost: 

Cost of executing this over King County.

- **>$1.5**

Cost of executing this over Kirkland.

- **>$0.4**

Make sure to filter out the unnecessary rasters outside of King County.


```python
import wkls

_aoi = wkls.us.wa.kingcounty.wkt()
```

In [2]:
import wkls


_aoi = wkls.us.wa.kirkland.wkt()

# Small bounding box in central Kirkland (~0.5km²)
kirkland_small = "POLYGON((-122.212 47.676, -122.200 47.676, -122.200 47.682, -122.212 47.682, -122.212 47.676))"

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [3]:
house_sales_df = (
    sedona.table(f"org_catalog.gde_bronze.king_co_homes")
            .where(f"ST_Intersects(geometry, ST_GeomFromWKT('{kirkland_small}'))")
            .withColumn("geometry_buffer", expr("ST_Buffer(geometry, 500, true)"))
)


house_sales_df.createOrReplaceTempView("house_sales")
house_sales_df.show()

[Stage 3:>                                                          (0 + 1) / 1]

+------+-----------+------------+----------+----------+--------+------------+-----------+---------+----------------+-----------------+----+--------+------+------------+-----------+--------+-------+----------+---------+--------+----+------+----------+-----+-----------+---------+-------+----+---------+---------+---------+---------+---------+----+----+---------+-------------+------------+-------------+-------------+----------------+------------+----------+-------------+-------------+---------------+----------+---------+--------------------+--------------------+
|   _c0|    sale_id|        pinx| sale_date|sale_price|sale_nbr|sale_warning|join_status|join_year|        latitude|        longitude|area|    city|zoning| subdivision|present_use|land_val|imp_val|year_built|year_reno|sqft_lot|sqft|sqft_1|sqft_fbsmt|grade|fbsmt_grade|condition|stories|beds|bath_full|bath_3qtr|bath_half|garb_sqft|gara_sqft|wfnt|golf|greenbelt|noise_traffic|view_rainier|view_olympics|view_cascades|view_territorial|vi

In [4]:
elevation_raster = (
    sedona.table("org_catalog.gde_bronze.elevation_bronze")
        .where(f"RS_Intersects(rast, ST_GeomFromWKT('{kirkland_small}'))")
)

elevation_raster.createOrReplaceTempView("elevation_raster")

# Calculate zonal stats on the elevation raster over the AOI

The AOIs are the house buffered polygon.

In [5]:
zonal_stats = sedona.sql(f"""

WITH zonal AS (
SELECT 
    RS_ZonalStatsAll(rast, geometry_buffer) AS zonal_stats,
    h.*
FROM
    house_sales h
JOIN
    elevation_raster e
ON
    RS_Intersects(h.geometry_buffer, e.rast)
)

SELECT
    *,
    zonal_stats.min as elevation_min,
    zonal_stats.max as elevation_max,
    zonal_stats.mean as elevation_mean,
    zonal_stats.stddev as elevation_standard_deviation
FROM
    zonal

""").cache()

zonal_stats.show()

[Stage 7:=======================================>                   (2 + 1) / 3]

+--------------------+------+-----------+------------+----------+----------+--------+------------+-----------+---------+----------------+-----------------+----+--------+------+------------+-----------+--------+-------+----------+---------+--------+----+------+----------+-----+-----------+---------+-------+----+---------+---------+---------+---------+---------+----+----+---------+-------------+------------+-------------+-------------+----------------+------------+----------+-------------+-------------+---------------+----------+---------+--------------------+--------------------+-----------------+-----------------+------------------+----------------------------+
|         zonal_stats|   _c0|    sale_id|        pinx| sale_date|sale_price|sale_nbr|sale_warning|join_status|join_year|        latitude|        longitude|area|    city|zoning| subdivision|present_use|land_val|imp_val|year_built|year_reno|sqft_lot|sqft|sqft_1|sqft_fbsmt|grade|fbsmt_grade|condition|stories|beds|bath_full|bath_3qt

In [6]:
(zonal_stats.select("sale_id", "elevation_min", "elevation_max", "elevation_mean", "elevation_standard_deviation")
        .writeTo("org_catalog.gde_silver.house_sales_zonal_silver")
        .createOrReplace()
)

# TRI enrichment

Using the same bronze `king_co_homes` table's dataframe `house_sales_df`.

## Clipping the elevation raster to AOI

Over here the AOI is the house buffered polygons.

In [7]:
elevation_raster_clipped = sedona.sql(f"""

SELECT 
    h.*,
    RS_Clip(
        rast,
        1,
        geometry_buffer
    ) AS clipped_rast
FROM
    elevation_raster e
JOIN
    house_sales h
ON
    RS_Intersects(e.rast, h.geometry_buffer)

""")

elevation_raster_clipped.createOrReplaceTempView("elevation_raster_clipped")

## Compute TRI rasters with clipped elevation rasters

TRI rasters are generated using a kernel size of 3x3. The TRI is defined as:


![image.png](https://raw.githubusercontent.com/wherobots/geospatial-data-engineering-associate/refs/heads/week-3/week-3/assets/tri_formula.png)
![image.png](https://raw.githubusercontent.com/wherobots/geospatial-data-engineering-associate/refs/heads/week-3/week-3/assets/tri_cal_representation.png)


where xi refers to each of the eight neighbors of the center cell E.

In [8]:
tri_raster = sedona.sql('''
    SELECT
        *,
        RS_MapAlgebra(
            clipped_rast,
            'D',
            '
            if (x() > 0 && x() < width() - 1 && y() > 0 && y() < height() - 1) {
                // Calculate the TRI for each cell that has valid neighbors
                out = (abs(rast - rast[-1, 0]) + abs(rast - rast[1, 0]) + 
                       abs(rast - rast[0, -1]) + abs(rast - rast[0, 1]) + 
                       abs(rast - rast[-1, -1]) + abs(rast - rast[-1, 1]) + 
                       abs(rast - rast[1, -1]) + abs(rast - rast[1, 1])) / 8;
            } else {
                out = 0;  // Set a default or "no data" value for edge cells
            }
            '
        ) AS tri_raster
    FROM elevation_raster_clipped
''')

tri_raster.createOrReplaceTempView("tri_raster")

In [9]:
tri_enriched = sedona.sql("""

SELECT 
    *,
    RS_SummaryStats(tri_raster, 'mean') as mean_tri

FROM tri_raster

""").select("sale_id", "mean_tri").cache()

tri_enriched.show()

[Stage 13:======================================>                   (2 + 1) / 3]

+-----------+------------------+
|    sale_id|          mean_tri|
+-----------+------------------+
|  2018..228|3.0944099255233275|
| 2018..2927| 3.119511866125876|
|2018..27078| 3.485553435949029|
| 2019..2060| 3.045378378208931|
| 2019..2121|3.3991241507312697|
| 2019..3008| 3.007798218283469|
| 2019..9147|3.2502590632025594|
|2019..12851|3.1987659815438723|
|2019..13837|2.2749074925037065|
|2019..16510| 2.941894297475932|
|2019..21345| 2.936981686452416|
|2019..23065| 3.143180844377326|
| 2020..6762|  2.70557716287946|
| 2020..6854| 3.110963914932484|
|2020..15225| 2.672437877660909|
|2005..19365|  2.70557716287946|
|2005..28171|3.1585211716637156|
|2005..31732|2.9979615781058326|
|2005..55198|3.0309741938529324|
| 2006..2068| 3.390303140284759|
+-----------+------------------+
only showing top 20 rows


In [10]:
tri_enriched.writeTo("org_catalog.gde_silver.house_sales_tri_silver").createOrReplace()

In [11]:
sedona.sql("SHOW TABLES IN org_catalog.gde_silver").show()

+----------+--------------------+-----------+
| namespace|           tableName|isTemporary|
+----------+--------------------+-----------+
|gde_silver|         census_data|      false|
|gde_silver|homes_distance_to...|      false|
|gde_silver|homes_distance_to...|      false|
|gde_silver| homes_flood_hazards|      false|
|gde_silver| homes_school_scores|      false|
|gde_silver|homes_seismic_haz...|      false|
|gde_silver|house_sales_ndvi_...|      false|
|gde_silver|house_sales_tri_s...|      false|
|gde_silver|house_sales_zonal...|      false|
|gde_silver|     roads_proximity|      false|
|gde_silver|         school_data|      false|
+----------+--------------------+-----------+

